# Private video-transfer Optuna screen
VideoMAE + 3D Swin. No submission. Report-proxy tuning only; no gold evaluation.


In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, subprocess, sys
ROOT = Path('/kaggle/working/transfer_optuna')
ROOT.mkdir(exist_ok=True)
FILES = {'models.py': '"""Local weights only; no remote model code or silent random initialization."""\nimport hashlib\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nVIDEOMAE_SHA = "065cfde56f97671da4d196e1a3a6d3cf9304dcad92eb671e3632a6c6cdd04f73"\nVIDEOMAE_CONFIG_SHA = "6d3c7afb3d1430fa478757425cc9be2fcc50a17bfbf58921e8a9e65bb3b5c568"\nSWIN_SHA = "7615ae035996b65eb38dad437ae533d2dfcd36f9f89d28c0f0fa7bfb8e6b3130"\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as stream:\n        for block in iter(lambda: stream.read(8 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef checked_asset(asset, filename=None):\n    path = Path(asset["path"])\n    hashed = path / filename if filename else path\n    if len(asset["sha256"]) != 64 or sha256(hashed) != asset["sha256"]:\n        raise ValueError(f"asset identity mismatch: {hashed}")\n    return path\n\n\ndef prepare_clip(pixels):\n    """[12,H,W] uint8 -> [3,16,224,224]; explicit repeated-slice adapter."""\n    if pixels.ndim != 3 or pixels.shape[0] != 12 or pixels.dtype != torch.uint8:\n        raise ValueError("expected twelve uint8 physical slices")\n    indices = torch.linspace(0, 11, 16, device=pixels.device).round().long()\n    x = pixels[indices, None].float() / 255\n    x = F.interpolate(x, size=(256, 256), mode="bilinear", align_corners=False,\n                      antialias=True)[:, :, 16:240, 16:240].expand(-1, 3, -1, -1)\n    mean = x.new_tensor([.485, .456, .406])[None, :, None, None]\n    std = x.new_tensor([.229, .224, .225])[None, :, None, None]\n    return ((x - mean) / std).permute(1, 0, 2, 3).contiguous()\n\n\ndef migrate_video_biases(state, expected):\n    """Exact old HF separate Q/V bias -> new Linear Q/K/V bias layout.\n\n    Old keys have no K bias: zero is algebraically identical, not an initialized\n    replacement parameter. No other checkpoint keys may be discarded or added.\n    """\n    state = dict(state)\n    for key in list(state):\n        if not key.endswith(".attention.attention.q_bias"):\n            continue\n        prefix = key.removesuffix("q_bias")\n        if prefix + "query.bias" not in expected:\n            continue  # Older transformers use the released layout directly.\n        q, v = state.pop(key), state.pop(prefix + "v_bias")\n        state[prefix + "query.bias"] = q\n        state[prefix + "value.bias"] = v\n        if prefix + "key.bias" in expected:\n            state[prefix + "key.bias"] = torch.zeros_like(q)\n    if set(state) != set(expected):\n        raise ValueError(f"VideoMAE state mismatch: missing={set(expected)-set(state)}, extra={set(state)-set(expected)}")\n    return state\n\n\nclass Reader(nn.Module):\n    def __init__(self, family, assets):\n        super().__init__()\n        self.family = family\n        if family == "swin3d_t":\n            from torchvision.models.video import swin3d_t\n            path = checked_asset(assets[family])\n            if assets[family]["sha256"] != SWIN_SHA:\n                raise ValueError("wrong Swin release")\n            self.encoder = swin3d_t(weights=None)\n            self.encoder.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True)\n            self.encoder.head = nn.Identity()\n            for p in self.encoder.parameters():\n                p.requires_grad_(False)\n            self.adapt = self.encoder.features[-1]\n            for p in list(self.adapt.parameters()) + list(self.encoder.norm.parameters()):\n                p.requires_grad_(True)\n        elif family == "videomae":\n            from safetensors.torch import load_file\n            from transformers import VideoMAEConfig, VideoMAEForVideoClassification\n            path = checked_asset(assets[family], "model.safetensors")\n            if assets[family]["sha256"] != VIDEOMAE_SHA:\n                raise ValueError("wrong VideoMAE release")\n            if assets[family].get("revision") != "488eb9a0565f257b32866000305c8178965eb9f6":\n                raise ValueError("wrong VideoMAE source revision")\n            if assets[family]["config_sha256"] != VIDEOMAE_CONFIG_SHA or sha256(path / "config.json") != VIDEOMAE_CONFIG_SHA:\n                raise ValueError("VideoMAE config drift")\n            config = VideoMAEConfig.from_pretrained(path, local_files_only=True)\n            original = VideoMAEForVideoClassification(config)\n            state = migrate_video_biases(load_file(path / "model.safetensors"), original.state_dict())\n            original.load_state_dict(state, strict=True)\n            if original.config.num_frames != 16 or original.config.hidden_size != 768:\n                raise ValueError("VideoMAE config drift")\n            self.encoder = original.videomae\n            self.final_norm = original.fc_norm\n            for p in self.encoder.parameters():\n                p.requires_grad_(False)\n            self.adapt = self.encoder.encoder.layer[-2:]\n            for p in self.adapt.parameters():\n                p.requires_grad_(True)\n        else:\n            raise ValueError(f"unknown model: {family}")\n        self.slot = nn.Parameter(torch.zeros(6, 768))\n        self.query = nn.Parameter(torch.randn(12, 768) * .02)\n        self.weight = nn.Parameter(torch.randn(12, 768) * .02)\n        self.bias = nn.Parameter(torch.zeros(12))\n\n    def train(self, mode=True):\n        super().train(mode)\n        self.encoder.eval()\n        self.adapt.train(mode)\n        return self\n\n    def encode(self, clip):\n        if self.family == "swin3d_t":\n            return self.encoder(clip)\n        tokens = self.encoder(pixel_values=clip.permute(0, 2, 1, 3, 4)).last_hidden_state\n        return self.final_norm(tokens.mean(1))\n\n    def forward(self, clips, mask):\n        # Sequential encoding bounds per-forward activation memory. Batch=1 study.\n        if clips.shape[:2] != (1, 6) or mask.shape != (1, 6) or not mask.any():\n            raise ValueError("expected a nonempty single-study six-slot input")\n        tokens = []\n        for slot in range(6):\n            if mask[0, slot]:\n                tokens.append(self.encode(clips[:, slot]) + self.slot[slot])\n            else:\n                tokens.append(self.slot[slot][None] * 0)\n        token = torch.stack(tokens, 1)\n        score = torch.einsum("bsd,td->bts", token, self.query) / 768 ** .5\n        score = score.masked_fill(~mask[:, None], -1e4)\n        context = torch.einsum("bts,bsd->btd", score.softmax(-1), token)\n        return (context * self.weight).sum(-1) / 768 ** .5 + self.bias\n', 'search.py': '"""Optuna screening on report-group-held-out pseudo labels, never organizer gold."""\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport optuna\nimport pandas as pd\nimport torch\nfrom sklearn.model_selection import GroupShuffleSplit\nfrom torch.nn import functional as F\n\nfrom models import Reader, checked_asset, prepare_clip, sha256\n\nTARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",\n           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nH5_SHA = "490e5ccc07143699d05a9eb3ef9583e08795f1c2bacc6d9cf2bd0a9268158526"\nSTUDIES_SHA = "efc63afee30345f94a916f98ba6b30f266fccfe2976b8475fa86653b2c510513"\nSEED, MAX_TRIALS, EPOCHS, BUDGET = 20261019, 8, 1, 7200\n\n\ndef groups_from_reports(reports):\n    groups = []\n    for report in reports:\n        if not isinstance(report, str) or not report.strip():\n            raise ValueError("missing report: cannot establish group exclusion")\n        normalized = " ".join(report.casefold().split())\n        groups.append(hashlib.sha256(normalized.encode("utf-8")).hexdigest())\n    return np.asarray(groups)\n\n\ndef partition(ids, groups, gold):\n    if len(set(ids)) != len(ids) or len(groups) != len(ids) or len(gold) != len(ids):\n        raise ValueError("identity/length drift")\n    allowed = np.flatnonzero(~gold & ~np.isin(groups, groups[gold]))\n    outer = GroupShuffleSplit(n_splits=1, test_size=.20, random_state=SEED)\n    development, confirmation = next(outer.split(allowed, groups=groups[allowed]))\n    development, confirmation = allowed[development], allowed[confirmation]\n    inner = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=SEED + 1)\n    fitting, tuning = next(inner.split(development, groups=groups[development]))\n    result = {"fit": development[fitting], "tune": development[tuning], "confirmation": confirmation}\n    names = list(result)\n    for i, name in enumerate(names):\n        if set(groups[result[name]]) & set(groups[gold]):\n            raise ValueError("gold report exposure")\n        for other in names[i + 1:]:\n            if set(groups[result[name]]) & set(groups[result[other]]):\n                raise ValueError("report group overlap")\n    return result\n\n\ndef validated_labels(train, labels):\n    ids = train.StudyInstanceUID.astype(str).to_numpy()\n    gold = train[TARGETS].notna().all(axis=1).to_numpy()\n    if len(ids) != 4407 or len(set(ids)) != 4407 or gold.sum() != 58:\n        raise ValueError("official competition identity/count drift")\n    aligned = train[["StudyInstanceUID"]].merge(labels, on="StudyInstanceUID", how="left", validate="one_to_one")\n    # Do not read organizer targets; these are only report-derived probabilities.\n    proxy = aligned[TARGETS].to_numpy(np.float32)\n    if not np.isfinite(proxy[~gold]).all() or (proxy[~gold] < 0).any() or (proxy[~gold] > 1).any():\n        raise ValueError("invalid non-gold proxy labels")\n    proxy[gold] = np.nan\n    return ids, gold, proxy\n\n\nclass Pixels:\n    def __init__(self, handle, order):\n        self.handle, self.order = handle, order\n\n    def get(self, index):\n        row = self.order[index]\n        mask = np.asarray(self.handle["masks"][row]).astype(bool)\n        if mask.shape != (6,) or not mask.any():\n            raise ValueError("missing study pixels; no fallback")\n        source = np.asarray(self.handle["pixels"][row])\n        clips = [prepare_clip(torch.from_numpy(source[s])) if mask[s] else torch.zeros(3, 16, 224, 224)\n                 for s in range(6)]\n        return torch.stack(clips)[None].cuda(), torch.from_numpy(mask)[None].cuda()\n\n\ndef tune_bce(model, pixels, indices, labels, deadline):\n    model.eval(); total = 0.\n    with torch.inference_mode():\n        for index in indices:\n            if time.monotonic() >= deadline:\n                raise TimeoutError("hard research budget reached during validation")\n            x, mask = pixels.get(index)\n            with torch.autocast("cuda", dtype=torch.float16):\n                logits = model(x, mask)\n            loss = F.binary_cross_entropy_with_logits(logits.float(), torch.from_numpy(labels[index:index+1]).cuda())\n            if not torch.isfinite(loss):\n                raise ValueError("nonfinite validation loss")\n            total += loss.item()\n    return total / len(indices)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--assets", required=True)\n    parser.add_argument("--out", required=True)\n    args = parser.parse_args()\n    import h5py\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA required; no CPU fallback")\n    assets_path = Path(args.assets)\n    assets = json.loads(assets_path.read_text())\n    if assets.get("terms_reviewed") is not True:\n        raise ValueError("complete the model terms review before research training")\n    if assets["h5"]["sha256"] != H5_SHA or assets["studies"]["sha256"] != STUDIES_SHA:\n        raise ValueError("wrong physical cache release")\n    paths = {key: checked_asset(assets[key]) for key in ("h5", "studies", "train", "labels")}\n    # Mandatory completed hardware preflight for these exact weights.\n    preflight = json.loads(checked_asset(assets["preflight"]).read_text())\n    if not preflight.get("synthetic_only") or not all(r["passed"] for r in preflight["models"]):\n        raise ValueError("both model execution preflights must pass")\n    if {r["family"] for r in preflight["models"]} != {"swin3d_t", "videomae"}:\n        raise ValueError("preflight model identity drift")\n    if preflight.get("weight_hashes") != {k: assets[k]["sha256"] for k in ("swin3d_t", "videomae")}:\n        raise ValueError("preflight checkpoint mismatch")\n    if preflight.get("gpu") != torch.cuda.get_device_name():\n        raise ValueError("rerun preflight on the actual training GPU; do not extrapolate laptop timings")\n    if preflight.get("models_source_sha256") != sha256(Path(__file__).with_name("models.py")):\n        raise ValueError("rerun preflight after changing model code")\n    train = pd.read_csv(paths["train"], dtype={"StudyInstanceUID": str})\n    labels = pd.read_csv(paths["labels"], dtype={"StudyInstanceUID": str})\n    ids, gold, proxy = validated_labels(train, labels)\n    groups = groups_from_reports(train.Report.tolist())\n    splits = partition(ids, groups, gold)\n    cache_ids = pd.read_csv(paths["studies"], dtype=str).StudyInstanceUID.tolist()\n    if len(cache_ids) != 4407 or len(set(cache_ids)) != 4407 or set(cache_ids) != set(ids):\n        raise ValueError("cache study identity drift")\n    row = {uid: i for i, uid in enumerate(cache_ids)}\n    order = np.asarray([row[uid] for uid in ids])\n    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)\n    contract = {"assets_sha256": sha256(assets_path), "source_sha256": sha256(__file__),\n                "models_source_sha256": sha256(Path(__file__).with_name("models.py")),\n                "seed": SEED, "epochs": EPOCHS, "trials": MAX_TRIALS, "budget_seconds": BUDGET,\n                "objective": "negative soft report-label BCE, not pathology AUC",\n                "gold_excluded": ids[gold].tolist(),\n                "splits": {k: ids[v].tolist() for k, v in splits.items()},\n                "gold_report_duplicate_exclusions": int((~gold & np.isin(groups, groups[gold])).sum())}\n    contract_path = out / "contract.json"\n    if contract_path.exists() and json.loads(contract_path.read_text()) != contract:\n        raise ValueError("cannot resume changed inputs/splits/source/search")\n    contract_path.write_text(json.dumps(contract, indent=2) + "\\n")\n    baseline_forecast = sum(max(r["step_seconds"][1:]) for r in preflight["models"]) * (\n        len(splits["fit"]) + 2 * len(splits["tune"])) * EPOCHS * 1.2\n    if baseline_forecast > BUDGET:\n        (out / "budget_rejection.json").write_text(json.dumps({\n            "reason": "both baseline architectures cannot fit measured budget",\n            "forecast_seconds": baseline_forecast, "budget_seconds": BUDGET,\n            "predictive_trials": 0, "gold_evaluations": 0}, indent=2) + "\\n")\n        raise RuntimeError("budget infeasible: no training launched; do not increase after inspecting labels")\n    study = optuna.create_study(direction="maximize", study_name="video-transfer-v1",\n                                storage=f"sqlite:///{(out / \'study.sqlite3\').resolve().as_posix()}",\n                                load_if_exists=True,\n                                sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=2),\n                                pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=0))\n    # In-flight/failed prior runs consume their maximum lease, not zero seconds.\n    consumed = sum(t.user_attrs.get("seconds", t.user_attrs.get("lease_seconds", 0)) for t in study.trials)\n    remaining_trials = MAX_TRIALS - len(study.trials)\n    if consumed >= BUDGET or remaining_trials <= 0:\n        raise RuntimeError("fixed search budget exhausted; do not extend after results")\n    deadline = time.monotonic() + BUDGET - consumed\n    if not study.trials:\n        for family in ("swin3d_t", "videomae"):\n            study.enqueue_trial({"family": family, "lr": 3e-5, "weight_decay": 1e-3})\n    with h5py.File(paths["h5"], "r") as handle:\n        if handle["pixels"].shape != (4407, 6, 12, 288, 288) or handle["pixels"].dtype != np.uint8:\n            raise ValueError("HDF5 pixel schema drift")\n        masks = np.asarray(handle["masks"])\n        if masks.shape != (4407, 6) or not np.isin(masks, (0, 1)).all() or masks.sum() != 23089:\n            raise ValueError("HDF5 mask identity drift")\n        pixels = Pixels(handle, order)\n\n        def objective(trial):\n            started = time.monotonic()\n            trial.set_user_attr("lease_seconds", max(0, deadline - started))\n            family = trial.suggest_categorical("family", ["swin3d_t", "videomae"])\n            lr = trial.suggest_float("lr", 1e-5, 1e-4, log=True)\n            wd = trial.suggest_float("weight_decay", 1e-4, 1e-2, log=True)\n            # Conservative all-slot compute forecast, before doing any training.\n            timing = next(r for r in preflight["models"] if r["family"] == family)\n            estimate = max(timing["step_seconds"][1:]) * (len(splits["fit"]) + 2 * len(splits["tune"]))\n            if estimate * EPOCHS * 1.2 > deadline - started:\n                trial.set_user_attr("seconds", time.monotonic() - started)\n                raise optuna.TrialPruned("not enough measured budget for a complete one-epoch screen")\n            random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)\n            model = Reader(family, assets).cuda()\n            optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=wd)\n            scaler = torch.amp.GradScaler("cuda")\n            rng = np.random.default_rng(SEED)\n            try:\n                for epoch in range(EPOCHS):\n                    epoch_start = time.monotonic(); model.train()\n                    for seen, index in enumerate(rng.permutation(splits["fit"]), 1):\n                        if time.monotonic() >= deadline:\n                            raise TimeoutError("hard study budget reached; no extra training")\n                        x, mask = pixels.get(index)\n                        optimizer.zero_grad(set_to_none=True)\n                        with torch.autocast("cuda", dtype=torch.float16):\n                            logits = model(x, mask)\n                        loss = F.binary_cross_entropy_with_logits(logits.float(), torch.from_numpy(proxy[index:index+1]).cuda())\n                        if not torch.isfinite(loss):\n                            raise ValueError("nonfinite training loss")\n                        scaler.scale(loss).backward(); scaler.unscale_(optimizer)\n                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)\n                        scaler.step(optimizer); scaler.update()\n                        if seen == (len(splits["fit"]) + 1) // 2:\n                            halfway_bce = tune_bce(model, pixels, splits["tune"], proxy, deadline)\n                            trial.report(-halfway_bce, 0)\n                            trial.set_user_attr("seconds", time.monotonic() - started)\n                            if trial.should_prune():\n                                raise optuna.TrialPruned("halfway proxy-BCE median pruning")\n                            model.train()\n                    bce = tune_bce(model, pixels, splits["tune"], proxy, deadline)\n                    checkpoint = out / f"trial_{trial.number}_latest.pt"\n                    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),\n                                "scaler": scaler.state_dict(), "epoch": epoch + 1, "params": trial.params,\n                                "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all(),\n                                "numpy_rng": rng.bit_generator.state, "python_rng": random.getstate(),\n                                "contract_sha256": sha256(contract_path)}, checkpoint)\n                    trial.set_user_attr("checkpoint_sha256", sha256(checkpoint))\n                    trial.set_user_attr("completed_epochs", epoch + 1)\n                    trial.set_user_attr("seconds", time.monotonic() - started)\n                    print(json.dumps({"trial": trial.number, "family": family, "epoch": epoch + 1, "proxy_bce": bce}), flush=True)\n                    trial.report(-bce, epoch + 1)\n                    if trial.should_prune():\n                        raise optuna.TrialPruned("median pruning on tuning groups only")\n                    if epoch + 1 < EPOCHS and (time.monotonic() - epoch_start) * 1.2 > deadline - time.monotonic():\n                        raise optuna.TrialPruned("measured real epoch will not fit remaining budget")\n                return -bce\n            finally:\n                trial.set_user_attr("seconds", time.monotonic() - started)\n                del model, optimizer\n                gc.collect(); torch.cuda.empty_cache()\n\n        study.optimize(objective, n_trials=remaining_trials, timeout=max(0, deadline - time.monotonic()))\n    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]\n    report = {"completed_trials": len(completed), "gold_evaluations": 0, "confirmation_evaluations": 0,\n              "submission_created": False, "claim": "proxy screening only; no model improvement established",\n              "selected": study.best_trial.params if completed else None,\n              "proxy_bce": -study.best_value if completed else None}\n    (out / "screen_report.json").write_text(json.dumps(report, indent=2) + "\\n")\n    print(json.dumps(report, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'preflight.py': '"""Bounded synthetic GPU execution check. Not a scientific validation."""\nimport argparse\nimport gc\nimport json\nimport time\nfrom pathlib import Path\n\nimport torch\nfrom models import Reader, sha256\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--assets", required=True)\n    parser.add_argument("--out", required=True)\n    args = parser.parse_args()\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA required; no CPU fallback")\n    assets = json.loads(Path(args.assets).read_text())\n    rows = []\n    for family in ("swin3d_t", "videomae"):\n        model = None\n        try:\n            torch.manual_seed(20261019)\n            model = Reader(family, assets).cuda().train()\n            optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-5)\n            x = torch.zeros(1, 6, 3, 16, 224, 224, device="cuda")\n            mask = torch.ones(1, 6, dtype=torch.bool, device="cuda")\n            torch.cuda.reset_peak_memory_stats()\n            times = []\n            for _ in range(3):\n                torch.cuda.synchronize(); start = time.monotonic()\n                optimizer.zero_grad(set_to_none=True)\n                # Full precision makes this a conservative execution check.\n                prediction = model(x, mask)\n                loss = prediction.float().square().mean()\n                loss.backward()\n                if not torch.isfinite(loss) or not all(\n                    torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None\n                ):\n                    raise ValueError("nonfinite synthetic gradient")\n                optimizer.step()\n                torch.cuda.synchronize(); times.append(time.monotonic() - start)\n            rows.append({"family": family, "passed": True, "step_seconds": times,\n                         "peak_bytes": torch.cuda.max_memory_allocated(),\n                         "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad)})\n        except Exception as error:\n            rows.append({"family": family, "passed": False,\n                         "error_type": type(error).__name__, "error": str(error)})\n        finally:\n            if model is not None:\n                del model\n            # Optimizer may own model tensors after a failed step.\n            if "optimizer" in locals():\n                del optimizer\n            gc.collect(); torch.cuda.empty_cache()\n    report = {"synthetic_only": True, "predictive_trials": 0,\n              "models_source_sha256": sha256(Path(__file__).with_name("models.py")),\n              "weight_hashes": {key: assets[key]["sha256"] for key in ("swin3d_t", "videomae")},\n              "gpu": torch.cuda.get_device_name(), "torch": torch.__version__, "models": rows}\n    Path(args.out).write_text(json.dumps(report, indent=2) + "\\n")\n    print(json.dumps(report, indent=2))\n    if not all(row["passed"] for row in rows):\n        raise SystemExit("Preflight failed: do not launch this two-model search")\n\n\nif __name__ == "__main__":\n    main()\n', 'fetch_preflight_assets.py': '"""Fetch two official releases for execution checks, not medical training data."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport torch\nfrom huggingface_hub import hf_hub_download\nfrom models import VIDEOMAE_SHA, sha256\n\nREVISION = "488eb9a0565f257b32866000305c8178965eb9f6"\nSWIN_URL = "https://download.pytorch.org/models/swin3d_t-7615ae03.pth"\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--out", required=True)\n    args = parser.parse_args()\n    out = Path(args.out).resolve(); out.mkdir(parents=True, exist_ok=True)\n    swin = out / "swin3d_t-7615ae03.pth"\n    if not swin.exists():\n        torch.hub.download_url_to_file(SWIN_URL, str(swin), hash_prefix="7615ae03")\n    if not sha256(swin).startswith("7615ae03"):\n        raise ValueError("official Swin filename hash mismatch")\n    video = out / "videomae"\n    for filename in ("config.json", "preprocessor_config.json", "model.safetensors", "README.md"):\n        hf_hub_download("MCG-NJU/videomae-base-finetuned-kinetics", filename,\n                        revision=REVISION, local_dir=video)\n    if sha256(video / "model.safetensors") != VIDEOMAE_SHA:\n        raise ValueError("VideoMAE weight identity mismatch")\n    config_sha = sha256(video / "config.json")\n    assets = {"terms_reviewed": False,\n              "purpose": "execution preflight only; not cleared for competition submission",\n              "swin3d_t": {"path": str(swin), "sha256": sha256(swin), "source": SWIN_URL},\n              "videomae": {"path": str(video), "sha256": VIDEOMAE_SHA,\n                           "config_sha256": config_sha, "revision": REVISION,\n                           "license": "CC-BY-NC-4.0"}}\n    (out / "assets.json").write_text(json.dumps(assets, indent=2) + "\\n")\n    print(json.dumps(assets, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, content in FILES.items():
    (ROOT / name).write_text(content)
# Do not change torch/torchvision or their CUDA builds.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'optuna==4.5.0', 'transformers==4.57.1', 'safetensors==0.6.2',
                'h5py==3.14.0'], check=True)
sys.path.insert(0, str(ROOT))
from models import sha256
subprocess.run([sys.executable, str(ROOT/'fetch_preflight_assets.py'),
                '--out', str(ROOT/'weights')], check=True)
assets_path = ROOT/'weights/assets.json'
subprocess.run([sys.executable, str(ROOT/'preflight.py'), '--assets', str(assets_path),
                '--out', str(ROOT/'preflight.json')], check=True)
assets = json.loads(assets_path.read_text())
def unique(name):
    hits = [p for p in Path('/kaggle/input').rglob(name) if p.is_file()]
    if len(hits) != 1:
        raise RuntimeError(f'expected unique {name}, found {hits}')
    return hits[0]
pins = {
    'h5': ('train_288x12.h5', '490e5ccc07143699d05a9eb3ef9583e08795f1c2bacc6d9cf2bd0a9268158526'),
    'studies': ('highres_cache_studies.csv', 'efc63afee30345f94a916f98ba6b30f266fccfe2976b8475fa86653b2c510513'),
    'train': ('train.csv', '8ca2203c0e9d61c080c7a314c7cdb51c1b03a1d9eb4770819f7f34af53ef4e33'),
    'labels': ('llm_labels_v4_blend.csv', 'eb4c2b29e7e84adbb4b5ccee825ca20ade6a43dbe295f13be442d549e4f5dca4'),
}
for key, (name, digest) in pins.items():
    assets[key] = {'path': str(unique(name)), 'sha256': digest}
assets['preflight'] = {'path': str(ROOT/'preflight.json'), 'sha256': sha256(ROOT/'preflight.json')}
# This is noncommercial private research only, not clearance for competition/prize use.
# Neither new encoder uses OAI weights. The OAI OrthoFoundation candidate is excluded.
assets['terms_reviewed'] = True
assets['terms_scope'] = 'private noncommercial no-submission research; VideoMAE CC-BY-NC-4.0'
assets_path.write_text(json.dumps(assets, indent=2) + '\n')
environment = {name: importlib.metadata.version(name) for name in
               ('torch', 'torchvision', 'transformers', 'optuna', 'h5py', 'safetensors')}
(ROOT/'environment.json').write_text(json.dumps(environment, indent=2) + '\n')
subprocess.run([sys.executable, str(ROOT/'search.py'), '--assets', str(assets_path),
                '--out', str(ROOT/'search-output')], check=True)
csvs = list(Path('/kaggle/working').rglob('*.csv'))
if csvs:
    raise RuntimeError(f'forbidden CSV output: {csvs}')
print('TRANSFER_OPTUNA_COMPLETE_NO_SUBMISSION', flush=True)
